In [1]:
from Montreal_UHI_toolbox import *
from sklearn.linear_model import LinearRegression
import plotly.express as px
template = 'plotly_white'

/runoff/gulley/.miniconda3/lib/python3.12/site-packages/gribapi/__init__.py:23: UserWarning: ecCodes 2.39.0 or higher is recommended. You are running version 2.14.1
  warnings.warn(


In [2]:
# Load all seasonal data from observation and otherwise into obs dataset
station_set = add_blurred_field_to_stations(static_fields_C['orog'],obs)
seasonal = {}
seasonal_std = {}
path = '/runoff/gulley/St_Laurent/intermediates'
# season = 'JJA'
for season in ['JJA','SON','DJF','MAM']:
    for field in ['tasmin','tasmax','tasavg']:

        # Loading simulated seasonal averages 
        seasonal[f'{field}_C'] = xr.open_zarr(f'{path}/sim/seasonal/seasons_avg_{field}_noTEB.zarr')[field].sel(season=season)
        seasonal[f'{field}_T'] = xr.open_zarr(f'{path}/sim/seasonal/seasons_avg_{field}_TEB.zarr')[field].sel(season=season)

        # Loading simulated seasonal standard deviations
        # seasonal_std[f'{field}_C'] = xr.open_zarr(f'{path}/sim/seasonal/seasons_std_{field}_noTEB.zarr')[field].sel(season=season)
        # seasonal_std[f'{field}_T'] = xr.open_zarr(f'{path}/sim/seasonal/seasons_std_{field}_TEB.zarr')[field].sel(season=season)

        
        # Adding smoothed simulated seasonal averages to nearest station
        station_set = add_blurred_field_to_stations(seasonal[f'{field}_C'],station_set=station_set,name=f'{field}_{season}_avg_C')
        station_set = add_blurred_field_to_stations(seasonal[f'{field}_T'],station_set=station_set,name=f'{field}_{season}_avg_T')

        # Adding smoothed seasonal standard deviation to nearest station
        # station_set = add_blurred_field_to_stations(seasonal_std[f'{field}_C'],station_set=station_set,name=f'{field}_{season}_std_C')
        # station_set = add_blurred_field_to_stations(seasonal_std[f'{field}_T'],station_set=station_set,name=f'{field}_{season}_std_T')

    # Loading station seasonal averages
    tasmax_S = xr.open_zarr(f'{path}/station/seasons_avg_tasmax.zarr')['tasmax'].sel(season=season) + 273.15
    tasmin_S = xr.open_zarr(f'{path}/station/seasons_avg_tasmin.zarr')['tasmin'].sel(season=season) + 273.15
    tasavg_S = xr.open_zarr(f'{path}/station/seasons_avg_tas.zarr')['tas'].sel(season=season).rename('tasavg') + 273.15
    # Loading station seasonal standard deviations
    # tasmax_std_S = xr.open_zarr(f'{path}/station/seasons_std_tasmax.zarr')['tasmax'].sel(season=season)
    # tasmin_std_S = xr.open_zarr(f'{path}/station/seasons_std_tasmin.zarr')['tasmin'].sel(season=season)
    # tasavg_std_S = xr.open_zarr(f'{path}/station/seasons_std_tas.zarr')['tas'].sel(season=season).rename('tasavg')

    for da in [tasmax_S,tasmin_S,tasavg_S]:
        station_set[f'{da.name}_{season}_avg_S'] = da

    # for da in [tasmax_std_S,tasmin_std_S,tasavg_std_S]:
    #     station_set[f'{da.name}_{season}_std_S'] = da

    station_set = station_set.dropna(dim="station", subset=[f'tasmax_{season}_avg_S',f'tasmin_{season}_avg_S',f'tasavg_{season}_avg_S'])


In [3]:
# season = 'JJA'
for season in ['JJA','SON','DJF','MAM']:
    for model_suffix, model_title in zip(['S','C','T'],['Observed','CLASS','TEB+CLASS']):
        fig = go.Figure()
        names = station_set.station_name.values
        x = station_set.urban_fraction_blurred_std1p5.values
        lat = station_set.lat.values

        # Creating the graphs, yavg is used to centre the y-axis
        yavg = (max(station_set[f'tasmax_{season}_avg_{model_suffix}'].values) + min(station_set[f'tasmin_{season}_avg_{model_suffix}'].values))/2 - 273.15
        for coord,colour,subscript,symbol,line_style in zip([f'tasmax_{season}_avg_{model_suffix}',f'tasavg_{season}_avg_{model_suffix}',f'tasmin_{season}_avg_{model_suffix}'],
        ['gold','orange','brown'],
        ['max','avg','min'],
        ['diamond', 'circle', 'square'],
        ['dash', 'dashdot', 'dot']):                

            # y = station_set[coord].values # original unadjusted for elevation
            
            # Adjust temperatures for station elevations
            Z_b = station_set.orog_blurred_std1p5.values + 2 # for temperatures coming from real elevations
            if model_suffix == 'S': # for temperatures coming from simulated elevations
                Z_b = station_set.elev.values
            y = [station_set[coord].values[i] + adjust_temp(station_set[coord].values[i],z_b=Z_b[i]) for i in range(len(y))] 
            y = y - np.ones(np.shape(y))*273.15
            var_name = f'<sup>{season}</sup><SPAN STYLE="text-decoration:overline">T</SPAN><sub>{subscript}</sub>'
            
            # Bin latitudes into N categories of latitude
            N = 5
            # edges = np.linspace(min(lat), max(lat), N + 1)
            edges = np.linspace(45, 46, N + 1)
            idx = np.digitize(lat, edges) - 1
            idx = np.clip(idx, 0, N - 1)

            # Get colorscale
            colors = px.colors.sample_colorscale('Bluered_r', np.linspace(0, 1, N))
            norm_edges = (edges - edges[0]) / (edges[-1] - edges[0])
            positions = np.repeat(norm_edges, 2)[1:-1]
            cs = [[p, c] for p, c in zip(positions, np.repeat(colors, 2))]
            lat_binned = (edges[idx] + edges[idx+1]) / 2

            # Scatter the stations by type, colour by latitude
            fig.add_trace(go.Scatter(
                x=x,
                y=y,
                mode='markers',
                text=names,
                textposition='top center',
                marker=dict(
                    size=8,
                    symbol=symbol,
                    color=lat_binned,          
                    colorscale=cs, # discrete colorscale
                    cmin=edges[0],
                    cmax=edges[-1],
                    colorbar=dict(
                        title='Latitude (°N)',
                        tickvals=(edges[:-1] + edges[1:]) / 2,
                        ticktext=[f"{a:.1f}–{b:.1f}" for a, b in zip(edges[:-1], edges[1:])],
                        xanchor='left',
                        y=0.3,
                        x = 1.,
                        len = 0.5,
                        # thickness=15,
                        # orientation='h'
                    )
                ),
                name=var_name,
                customdata=np.stack([lat], axis=-1),
                hovertemplate=(
                    f'%{{text}}'
                    f'<br>lat: %{{customdata[0]:.2f}}°'
                    f'<br>urban_blurred_std1p5: %{{x:.2f}}'
                    f'<br>{coord}: %{{y:.2f}}°C'
                    f'<extra></extra>'
                )
            ))

            # Linear Regression
            model = LinearRegression()
            X = x.reshape(-1, 1)
            model.fit(X,y)
            r_square = model.score(X,y)
            xmax = 0.7
            xline = np.linspace(0,xmax,2)
            yline = xline*model.coef_ + model.intercept_

            # Now to format best fit line equation for Latex
            line_label = f'y={model.coef_[0]:+.2f}x{model.intercept_:+.2f}, R^2={r_square:.2f}'
            line_label = f'${line_label}$'
            
            # Adding best-fit line
            # fig.add_trace(go.Scatter(
            #     x=xline,
            #     y=yline.flatten(),
            #     mode='lines',
            #     line=dict(color=colour,dash='dot'),
            #     name=line_label
            # ))
            
            fig.add_trace(go.Scatter(
                x=xline,
                y=yline.flatten(),
                mode='lines',
                line=dict(color='black', dash=line_style),
                name=line_label
            ))
            
            # Axes labels, title, limits
            ymin = yavg - 7
            ymax = yavg + 7
            fig.update_layout(
                title=f'{model_title} <sup>{season}</sup><SPAN STYLE="text-decoration:overline">T</SPAN><sub>daily</sub>',
                xaxis_title='Urban Fraction',
                yaxis_title=f'<sup>{season}</sup><SPAN STYLE="text-decoration:overline">T</SPAN><sub>daily</sub> (°C)',
                xaxis=dict(range=[-0.01, xmax]),
                yaxis=dict(range=[ymin,ymax]),
                margin=dict(r=120),
                template=template
            )

        # Where to write text for urban, rural, suburban
        urban_thresh = 0.5
        suburban_thresh = 0.2
        rural_thresh = 0.01

        # Drawing urban thresholds
        fig.add_vline(x=rural_thresh, line=dict(color='lightgrey', dash='dash'), annotation_text='Rural', annotation_position='top left',annotation_font=dict(color='grey'))
        fig.add_vline(x=urban_thresh, line=dict(color='lightgrey', dash='dash'), annotation_text='Urban', annotation_position='top right',annotation_font=dict(color='grey'))
        suburban_x = (rural_thresh + urban_thresh) / 2
        fig.add_annotation(
            x=suburban_x,
            y=1.0,
            yref='paper',
            text='Suburban',
            showarrow=False,
            align='center',
            font=dict(color='grey')
        )
        
        # Display figure
        fig.update_xaxes(dtick=0.1)
        # fig.show()
        fig.write_html(f'/runoff/gulley/UHI_plots/temps_vs_furban/adjtemp_vs_furban_{season}_{model_suffix}.html',include_mathjax='cdn')

NameError: name 'y' is not defined